# 记忆检索策略 (Memory Retrieval) 完整教程

## 概述

记忆检索是从长期记忆中找到最相关信息的关键环节。本教程介绍：

1. **检索信号** - 相似度、时效性、重要性
2. **时间衰减** - 指数衰减模型
3. **四种检索策略** - 各有侧重的实现
4. **混合检索** - 综合多信号的最佳实践

---

## 环境准备

In [ ]:
import sys
sys.path.insert(0, '../src')

from memory_retrieval import (
    TimeDecay, RetrievalResult,
    SimilarityRetrieval, RecencyRetrieval,
    ImportanceRetrieval, HybridRetrieval,
    MemoryRetriever
)
from long_term_memory import LongTermMemory, MemoryEntry, MemoryType
from datetime import datetime, timedelta
import math

print("模块导入成功!")

---

## 1. 时间衰减 (Time Decay)

### 1.1 指数衰减模型

$$\text{recency}(t) = e^{-\lambda \cdot \Delta t}$$

In [ ]:
# 创建不同衰减速度
no_decay = TimeDecay.no_decay()
slow_decay = TimeDecay.slow_decay()  # 24小时半衰期
fast_decay = TimeDecay.fast_decay()  # 1小时半衰期

# 测试不同时间点
now = datetime.now()
times = [
    ("刚刚", now),
    ("1小时前", now - timedelta(hours=1)),
    ("1天前", now - timedelta(days=1)),
    ("1周前", now - timedelta(weeks=1)),
]

print("=== 时间衰减对比 ===")
print(f"{'时间点':<10} | {'无衰减':>8} | {'慢衰减':>8} | {'快衰减':>8}")
print("-" * 45)
for name, t in times:
    print(f"{name:<10} | {no_decay.compute(t):>8.4f} | {slow_decay.compute(t):>8.4f} | {fast_decay.compute(t):>8.4f}")

### 1.2 自定义半衰期

In [ ]:
# 自定义半衰期
custom_decay = TimeDecay(half_life_hours=6.0)  # 6小时半衰期

six_hours_ago = datetime.now() - timedelta(hours=6)
score = custom_decay.compute(six_hours_ago)

print(f"6小时半衰期，6小时前的分数: {score:.4f}")
print(f"预期约为 0.5 (半衰期定义)")

---

## 2. 检索策略

### 2.1 准备测试数据

In [ ]:
# 创建长期记忆并添加测试数据
ltm = LongTermMemory()

# 添加不同特征的记忆
test_memories = [
    ("Python 是一种编程语言", MemoryType.KNOWLEDGE, 0.5, 0),
    ("用户喜欢 Python 编程", MemoryType.PREFERENCE, 0.8, 1),
    ("机器学习使用 Python", MemoryType.KNOWLEDGE, 0.6, 24),
    ("用户昨天学习了 Python", MemoryType.EVENT, 0.4, 48),
    ("重要: 用户是 Python 专家", MemoryType.FACT, 0.95, 72),
]

for content, mem_type, importance, hours_ago in test_memories:
    entry = ltm.store(content, memory_type=mem_type, importance=importance)
    # 修改时间戳模拟不同时间
    entry.timestamp = datetime.now() - timedelta(hours=hours_ago)

print(f"创建了 {ltm.size} 条测试记忆")

### 2.2 相似度检索 (SimilarityRetrieval)

In [ ]:
# 相似度检索
sim_strategy = SimilarityRetrieval()

# 获取原始结果
raw_results = ltm.recall("Python 编程", k=5)

# 使用策略排序
ranked = sim_strategy.rank(raw_results)

print("=== 相似度检索 ===")
print("只考虑语义相似度\n")
for r in ranked:
    print(f"  [{r.similarity:.3f}] {r.entry.content}")

### 2.3 时效性检索 (RecencyRetrieval)

In [ ]:
# 时效性检索
recency_strategy = RecencyRetrieval(time_decay=TimeDecay.slow_decay())

ranked = recency_strategy.rank(raw_results)

print("=== 时效性检索 ===")
print("只考虑时间新旧\n")
for r in ranked:
    age = (datetime.now() - r.entry.timestamp).total_seconds() / 3600
    print(f"  [{r.recency:.3f}] ({age:.0f}h前) {r.entry.content}")

### 2.4 重要性检索 (ImportanceRetrieval)

In [ ]:
# 重要性检索
importance_strategy = ImportanceRetrieval()

ranked = importance_strategy.rank(raw_results)

print("=== 重要性检索 ===")
print("只考虑重要性分数\n")
for r in ranked:
    print(f"  [{r.importance:.3f}] {r.entry.content}")

### 2.5 混合检索 (HybridRetrieval)

In [ ]:
# 混合检索 - 综合三个信号
hybrid_strategy = HybridRetrieval(
    alpha=0.5,  # 相似度权重
    beta=0.3,   # 时效性权重
    gamma=0.2   # 重要性权重
)

ranked = hybrid_strategy.rank(raw_results)

print("=== 混合检索 ===")
print("score = 0.5*sim + 0.3*recency + 0.2*importance\n")
for r in ranked:
    print(f"  [{r.score:.3f}] (sim={r.similarity:.2f}, rec={r.recency:.2f}, imp={r.importance:.2f})")
    print(f"           {r.entry.content}")

---

## 3. MemoryRetriever 高级接口

In [ ]:
# 创建检索器
retriever = MemoryRetriever(ltm, strategy=HybridRetrieval())

# 检索
results = retriever.retrieve("Python", k=3)

print("=== MemoryRetriever 检索 ===")
for r in results:
    print(f"  [{r.score:.3f}] {r.entry.content}")

In [ ]:
# 检索重要记忆
important = retriever.retrieve_important(k=3, min_importance=0.6)

print("=== 重要记忆 (importance >= 0.6) ===")
for r in important:
    print(f"  [{r.entry.importance:.2f}] {r.entry.content}")

---

## 4. 权重配置指南

| 场景 | alpha (相似度) | beta (时效) | gamma (重要性) |
|------|---------------|-------------|----------------|
| 知识问答 | 0.7 | 0.1 | 0.2 |
| 客服对话 | 0.5 | 0.3 | 0.2 |
| 个人助理 | 0.4 | 0.3 | 0.3 |
| 实时聊天 | 0.3 | 0.5 | 0.2 |

In [ ]:
# 不同场景配置对比
configs = {
    "知识问答": (0.7, 0.1, 0.2),
    "客服对话": (0.5, 0.3, 0.2),
    "实时聊天": (0.3, 0.5, 0.2),
}

print("=== 不同配置的检索结果 ===")
for name, (a, b, g) in configs.items():
    strategy = HybridRetrieval(alpha=a, beta=b, gamma=g)
    ranked = strategy.rank(raw_results)
    top = ranked[0] if ranked else None
    if top:
        print(f"\n{name} (a={a}, b={b}, g={g}):")
        print(f"  Top: {top.entry.content}")

---

## 总结

记忆检索策略的选择取决于应用场景：

- **SimilarityRetrieval**: 纯语义匹配，适合知识库
- **RecencyRetrieval**: 时间优先，适合实时对话
- **ImportanceRetrieval**: 重要性优先，适合关键信息
- **HybridRetrieval**: 综合考虑，适合通用场景

通过调整权重参数，可以针对不同场景优化检索效果。